In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

!pip install timm transformers scikit-learn tqdm opencv-python-headless -q

Mounted at /content/drive


In [ ]:
import sys

# Point this to wherever your repo src/ and models/ folders are in your Drive
REPO_PATH = '/content/drive/MyDrive/ADL Project/chestxray-classification-main'

# Add src and root to path so imports work
sys.path.insert(0, f'{REPO_PATH}/src')
sys.path.insert(0, REPO_PATH)

In [ ]:
# Update DATASET_ROOT to local SSD instead of Drive

BASE = "/content/drive/.shortcut-targets-by-id/156wJ6Q8hISoGtwsJK9BAZXAM283DjuwE/adl_cv_data"

DATASET_ROOT    = "/content/dataset"
CSV_PATH        = f"{BASE}/dataset/Data_Entry_2017.csv"
CHECKPOINT_ROOT = f"{BASE}/checkpoints"
SOURCE_ZIP      = f"{BASE}/data.zip"

CHECKPOINT_MAP = {
    'Swin Transformer': f'{CHECKPOINT_ROOT}/Swin Transformer/Copy of swin_best_model.pth',
    'ResNet50':         f'{CHECKPOINT_ROOT}/ResNet50/resnet50_best_model.pth',
    'RadJEPA':          f'{CHECKPOINT_ROOT}/RadJEPA/radjepa_best_model.pth',
    'RadDINO':          f'{CHECKPOINT_ROOT}/RadDINO/raddino_best_model.pth',
    'EfficientNet-B0':  f'{CHECKPOINT_ROOT}/EfficientNet-B0/efficientnet_best_model.pth',
    'ConvNeXt-V2':      f'{CHECKPOINT_ROOT}/ConvNeXt-V2/convnext_best_model.pth',
}

In [ ]:
import os
DATASET_LOCAL = "/content/dataset"

os.makedirs(DATASET_LOCAL, exist_ok=True)

print("Copying zip from Drive to local SSD...")
!cp "{SOURCE_ZIP}" /content/data_local.zip
print("Copy done. Unzipping...")

!unzip -q /content/data_local.zip -d "{DATASET_LOCAL}"

os.remove("/content/data_local.zip")
print("Done! Checking structure:")
!ls "{DATASET_LOCAL}" | head -20

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, f'{REPO_PATH}/src')
sys.path.insert(0, REPO_PATH)

from build_path_map import build_dataframe_with_paths
from data_split import split_data
from dataset_loader import NIHDataset
from torch.utils.data import DataLoader

# Build dataframe with image paths
df = build_dataframe_with_paths(CSV_PATH, DATASET_ROOT)

# Reproduce the exact same split (random_state=42 is hardcoded in your data_split.py)
train_df, val_df, test_df = split_data(df)
print(f"Val size: {len(val_df)} | Test size: {len(test_df)}")

# Create dataset and loader — use val set for evaluation
val_dataset = NIHDataset(val_df, is_train=False)
val_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

DISEASES = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass",
    "Nodule","Pneumonia","Pneumothorax","Consolidation","Edema",
    "Emphysema","Fibrosis","Pleural_Thickening","Hernia"
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def evaluate_model(model, loader, device):
    model.eval()
    all_probs  = []
    all_labels = []
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs   = torch.sigmoid(outputs)

            # Accuracy
            preds = (probs > 0.5).float()
            total_correct  += (preds == labels).float().mean(dim=1).sum().item()
            total_samples  += images.size(0)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.concatenate(all_probs,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    accuracy   = total_correct / total_samples

    # Per-class AUROC
    per_class_auroc = {}
    valid_scores    = []
    for i, disease in enumerate(DISEASES):
        if len(np.unique(all_labels[:, i])) > 1:
            score = roc_auc_score(all_labels[:, i], all_probs[:, i])
            per_class_auroc[disease] = round(score, 4)
            valid_scores.append(score)
        else:
            per_class_auroc[disease] = None  # not enough samples

    macro_auroc = float(np.mean(valid_scores)) if valid_scores else 0.5
    return accuracy, macro_auroc, per_class_auroc

In [ ]:
from models.generic_cv.ConvNeXt_V2.model    import ConvNeXtV2
from models.generic_cv.Swin_Transformer.model import SwinTransformer
from models.medical_sota.RadDINO.model      import RadDINO
from models.medical_sota.RadJEPA.model      import RadJEPA

NUM_CLASSES = 14

MODEL_CONSTRUCTORS = {
    'ConvNeXt-V2':      lambda: ConvNeXtV2(NUM_CLASSES),
    'Swin Transformer': lambda: SwinTransformer(NUM_CLASSES),
    'RadDINO':          lambda: RadDINO(NUM_CLASSES, freeze_backbone=True),
    'RadJEPA':          lambda: RadJEPA(NUM_CLASSES),
}

results = {}

for model_name, constructor in MODEL_CONSTRUCTORS.items():
    print(f"\n{'='*50}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*50}")

    # Build model and load weights
    model = constructor()
    checkpoint_path = CHECKPOINT_MAP[model_name]
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)

    # Evaluate
    accuracy, macro_auroc, per_class_auroc = evaluate_model(model, val_loader, device)

    results[model_name] = {
        'accuracy':       round(accuracy, 4),
        'macro_auroc':    round(macro_auroc, 4),
        'per_class_auroc': per_class_auroc,
    }

    print(f"  Accuracy:    {accuracy:.4f}")
    print(f"  Macro AUROC: {macro_auroc:.4f}")
    print(f"  Per-class AUROC:")
    for disease, score in per_class_auroc.items():
        print(f"    {disease:<20}: {score}")

    # Free GPU memory between models
    del model
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd

# Previously evaluated models
results['ResNet50'] = {
    'accuracy': 0.9490,
    'macro_auroc': 0.8387,
    'per_class_auroc': {
        'Atelectasis':        0.8089,
        'Cardiomegaly':       0.8923,
        'Effusion':           0.8826,
        'Infiltration':       0.7171,
        'Mass':               0.8442,
        'Nodule':             0.7601,
        'Pneumonia':          0.7489,
        'Pneumothorax':       0.8782,
        'Consolidation':      0.8146,
        'Edema':              0.8888,
        'Emphysema':          0.9147,
        'Fibrosis':           0.8154,
        'Pleural_Thickening': 0.8127,
        'Hernia':             0.9628,
    }
}

results['EfficientNet-B0'] = {
    'accuracy': 0.9477,
    'macro_auroc': 0.8300,
    'per_class_auroc': {
        'Atelectasis':        0.8066,
        'Cardiomegaly':       0.8975,
        'Effusion':           0.8767,
        'Infiltration':       0.7035,
        'Mass':               0.8322,
        'Nodule':             0.7483,
        'Pneumonia':          0.7460,
        'Pneumothorax':       0.8729,
        'Consolidation':      0.8104,
        'Edema':              0.8865,
        'Emphysema':          0.9191,
        'Fibrosis':           0.7940,
        'Pleural_Thickening': 0.7957,
        'Hernia':             0.9301,
    }
}

# Summary table
summary_rows = []
for model_name, r in results.items():
    row = {'Model': model_name, 'Accuracy': r['accuracy'], 'Macro AUROC': r['macro_auroc']}
    row.update({d: r['per_class_auroc'].get(d) for d in DISEASES})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print(summary_df.to_string())

# Save to CSV
# summary_df.to_csv('/content/drive/MyDrive/ADL Project/evaluation_results.csv')
summary_df.to_csv(f'{BASE}/evaluation_results.csv')
print("\nSaved to Drive!")